# 策略概述

**距離基準交易**以 Gatev, Goetzmann & Rouwenhorst (2006) 的**正規化價格距離**建構價差，取代回歸基準（Z-Score 版）以共整合殘差建構的價差。

兩者**共用同一套進出場狀態機**（訊號、風控、費用會計完全相同），唯一差異在「價差怎麼定義」——因此構成「回歸 vs 距離」的乾淨單變因對照（教授指定的消融實驗）。


# 策略架構

交易流程與 Z-Score 基準完全相同，**只替換第一個環節「價差的定義空間」**。

```{mermaid}
flowchart LR
  A["價差<br/>正規化價格距離（GGR）"] --> B["訊號<br/>偏離進場、回歸出場"]
  B --> C["部位<br/>等權配置"]
  C --> D["風控<br/>比例停損"]
  D --> E["期末<br/>強制結算"]
```

| 環節 | 本引擎作法 | 對照 |
| :--- | :--- | :--- |
| **價差重建** | **正規化價格距離** | ← 唯一差異（Z-Score 版用共整合殘差） |
| 訊號 | 偏離 2 倍標準差進場、回歸出場 | 與 Z-Score 版相同 |
| 部位 | 等權配置 | 與 Z-Score 版相同 |
| 風控 / 期末 | 比例停損、強制結算 | 與 Z-Score 版相同 |


## 回歸基準 vs 距離基準

| | 回歸基準（`zscore_trading`） | 距離基準（本模組） |
| :--- | :--- | :--- |
| spread 空間 | 共整合殘差：$\ln P_A - \alpha - \beta \ln P_B$ | 正規化價格距離：$\tilde P_A - \tilde P_B$ |
| 對沖比率 $\beta$ | OLS 回歸估計 | 固定 $= 1$（等權） |
| 訊號來源 | 殘差均值回歸 | 兩條正規化價格路徑的發散/收斂 |

除 spread 定義外，進場（$|z|>entry_z$）、出場（$z$ 穿越 $\pm exit_z$）、期末強平、風控與費用會計皆與回歸基準一致。


# 參考文獻與引用對應


## 文獻 1：Gatev, Goetzmann & Rouwenhorst (2006)

> Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative value arbitrage rule. *Review of Financial Studies*, **19**(3), 797–827.

**參考部分**：

- 距離法交易規則：以標準化價格的距離衡量偏離，偏離超過**形成期 2 倍標準差**開倉、收斂平倉、期末強平
- 等權對沖（$\beta=1$），不做回歸

**為何參考**：

- 本模組即此距離法交易規則的實作：spread 為正規化價格差、對沖固定 $\beta=1$、
  $|z|>2$ 開倉對應「2 倍歷史標準差」門檻



## 文獻 2：Do & Faff (2012)

> Do, B., & Faff, R. (2012). Are pairs trading profits robust to trading costs? *Journal of Financial Research*, **35**(2), 261–287.

**參考部分**：交易成本對配對交易獲利的侵蝕。

**為何參考**：每筆進出場扣往返成本（`fee_rate=0.0029`、`slippage=0`，往返 0.58%）之依據。



# 各階段行為


## 階段 1：距離 spread 建構（依據：Gatev 2006）

以形成期 $\mu$、$\sigma$ 標準化 log-price：

$$\tilde P_{A,t} = \frac{\ln P_{A,t} - \mu_A^{form}}{\sigma_A^{form}}, \qquad
\tilde P_{B,t} = \frac{\ln P_{B,t} - \mu_B^{form}}{\sigma_B^{form}}$$

距離 spread（等權、無回歸，對沖比率固定 $=1$）：

$$D_t = \tilde P_{A,t} - \tilde P_{B,t}$$


## 階段 2：Z-Score 與交易訊號

以**形成期距離 spread** 的均值與標準差標準化：

$$z_t = \text{clip}\left(\frac{D_t - \mu_D^{form}}{\sigma_D^{form}},\ -10,\ 10\right)$$

$\mu_D$、$\sigma_D$ 為形成期距離 spread 的統計量（GGR 的「2 個歷史標準差開倉」即 $z_D = 2$）。


## 階段 3：共用 Z-Score 狀態機

與回歸基準完全相同：

| 條件 | 動作 |
| :--- | :--- |
| $z_t > entry_z$（=2.0） | 空 A、多 B（風險中性配置，$\beta=1$ 故兩腳等額） |
| $z_t < -entry_z$ | 多 A、空 B |
| 空頭且 $z_t \le exit_z$／多頭且 $z_t \ge -exit_z$ | 平倉 |
| 期末仍持倉 | 強制結算 |

停損、凍結、費用會計（進出場各扣 $0.29\%$）皆沿用 `zscore_trading` 的六大風控機制。


# 參數總表

| 參數 | 值 | 對應環節 | 說明 |
| :--- | :---: | :--- | :--- |
| 交易期長度 | 126 交易日 | 全流程 | 滾動步長 21 日 |
| 進場 / 出場門檻 | 2.0 / 0.0 倍標準差 | 訊號 | GGR 標準設定 |
| 對沖比率 | 固定 1.0 | 部位配置 | 等權距離法 |
| 價差空間 | 正規化價格距離 | 價差重建 | 對照共整合殘差空間 |
| 往返交易成本 | 0.58% | 損益 | 與 Z-Score 版相同 |
| 比例停損 | 網格 [0, 5%, 15%] | 風控 | 單筆虧損上限 |
